# Expenses model

**Data**: `data-1786540369000.csv` holds two entities joined in one file:

| columns | entity | grain |
|---|---|---|
| `uuid`, `value`, `created`, `modified` | proponent snapshot as of now (full hAPI profile JSON) | one per proponent, repeated across join rows |
| `uuid-2`, `anonymized_training_data_uuid`, `field_name`, `year`, `month`, `value-2`, ... | proponent's CRC at T-1 month of each credit in the current CRC | one per (proponent, year, month) |

`anonymized_training_data_uuid` links the CRC history rows back to `uuid`.

This notebook unpacks the JSON dictionaries into three flat tables:
1. `prop`: one row per proponent, every nested dict flattened into columns (lists kept as list-columns for later).
2. `crc_hist`: one row per historical CRC pull (proponent, year, month).
3. `liab`: `crc_hist` exploded to one row per liability, all liability fields as columns.

In [ ]:
import json

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

raw = pd.read_csv("../data/data-1786540369000.csv")
print(f"as extracted: {len(raw)} rows, {raw['uuid'].nunique()} proponents")

# CRC history rows created before 2026-08-07 came from a bad extraction run:
# the CURRENT CRC was stored under historical (year, month) labels, leaking
# future credits. Keep only the clean re-extraction from Aug 7 on.
raw["created-2"] = pd.to_datetime(raw["created-2"])
raw = raw[raw["created-2"] >= pd.Timestamp("2026-08-07", tz="UTC")].reset_index(drop=True)

print(f"after filter:  {len(raw)} rows, {raw['uuid'].nunique()} proponents")
raw.head(2)

## 1. Proponent table: flatten the profile JSON

`pd.json_normalize` with `sep="_"` turns every nested dict into a column (`Income_Pension`, `PersonalTaxData_Gender`, `IRSTaxDeclarationDetails_Total`, ...).
Fields whose value is a **list** (`ExpensesList`, `NetSalary`, `CRCData_Liabilities`, ...) cannot become scalar columns at this grain; they stay as list-columns and get exploded into their own tables when we need them.

In [ ]:
prop_raw = (
    raw[["uuid", "value", "created", "modified"]]
    .drop_duplicates("uuid")
    .reset_index(drop=True)
)
profiles = prop_raw["value"].map(json.loads)

prop = pd.json_normalize(profiles, sep="_")
prop.insert(0, "uuid", prop_raw["uuid"])
prop.insert(1, "snapshot_created", pd.to_datetime(prop_raw["created"]))

list_cols = [c for c in prop.columns if prop[c].map(lambda x: isinstance(x, list)).any()]
print(f"prop: {prop.shape[0]} proponents x {prop.shape[1]} columns")
print(f"{len(list_cols)} list-columns kept for later explosion, e.g. {list_cols[:6]}")
prop.head(3)

In [ ]:
# Full column inventory of the flattened proponent table
scalar_cols = [c for c in prop.columns if c not in list_cols]
print(f"scalar columns ({len(scalar_cols)}):")
for c in scalar_cols:
    print("  ", c)
print(f"\nlist columns ({len(list_cols)}):")
for c in list_cols:
    n = prop[c].map(lambda x: len(x) if isinstance(x, list) else 0)
    print(f"   {c}  (avg {n.mean():.1f} items, max {n.max()})")

## 2. CRC history: one row per pull, then one row per liability

`value-2` is the CRC as of (`year`, `month`), the month before each credit's origination.
First flatten the envelope, then explode `Liabilities` so every liability becomes a row with proper columns.

In [ ]:
crc_raw = raw[
    ["uuid-2", "anonymized_training_data_uuid", "field_name", "year", "month", "value-2"]
].rename(columns={"uuid-2": "crc_uuid", "anonymized_training_data_uuid": "uuid"})

crc_hist = pd.json_normalize(crc_raw["value-2"].map(json.loads), sep="_")
crc_hist = pd.concat([crc_raw.drop(columns="value-2").reset_index(drop=True), crc_hist], axis=1)
print(f"crc_hist: {crc_hist.shape[0]} pulls x {crc_hist.shape[1]} columns")
crc_hist.head(3)

In [ ]:
liab = crc_hist.explode("Liabilities", ignore_index=True)
liab = pd.concat(
    [liab.drop(columns="Liabilities"), pd.json_normalize(liab["Liabilities"], sep="_")],
    axis=1,
)
print(f"liab: {liab.shape[0]} liabilities x {liab.shape[1]} columns")
liab[["uuid", "year", "month", "Product", "BeginDate", "TotalDebt", "Installment", "TotalDefault"]].head(10)

## 3. Full dataset: proponent + CRC in the past

`full` joins the past-CRC liabilities with everything we know about the proponent today, on `uuid`.
Grain stays one row per (liability, pull): proponent columns repeat across their liabilities.

In [ ]:
full = liab.merge(prop, on="uuid", how="left", suffixes=("", "_prop"))

assert len(full) == len(liab), "join must not duplicate liability rows"
print(f"full: {full.shape[0]} rows (liability x pull) x {full.shape[1]} columns")
print(f"proponents: {full['uuid'].nunique()}, pulls: {full['crc_uuid'].nunique()}")
full.head(3)

## 4. Expenses model

### 4.1 Target: recurring monthly total spend

One number per proponent, `monthly_expenses`, built from `ExpensesList`:

1. Explode receipts (one row per e-fatura record).
2. Drop **one-offs**: receipts >= 1000 EUR (durables: appliances, dental work, ...). They are real spend but not recurring cost.
3. `monthly_expenses` = kept spend / observed month span (first to last receipt month, so empty months count as zero).

Total spend across all categories, not essential-only: 89% of spend sits in `Outros`/null sector, so an essential-only figure would be a heavy undercount.

**Caveat**: e-fatura misses rent, most utilities, and loan payments, and only captures invoiced spend. The target is a lower bound on true outgoings, and the model inherits that meaning.

In [ ]:
# Receipts table: one row per e-fatura record
rec = (
    prop[["uuid", "ExpensesList"]]
    .explode("ExpensesList", ignore_index=True)
    .dropna(subset=["ExpensesList"])
)
rec = pd.concat(
    [rec[["uuid"]].reset_index(drop=True), pd.json_normalize(rec["ExpensesList"].tolist())],
    axis=1,
)
rec["Amount"] = pd.to_numeric(rec["Amount"], errors="coerce")
rec["month"] = (
    pd.to_datetime(rec["IssuanceDate"], utc=True, format="mixed")
    .dt.tz_localize(None)
    .dt.to_period("M")
)
print(f"rec: {len(rec)} receipts, {rec['uuid'].nunique()} proponents, "
      f"{rec['month'].min()} to {rec['month'].max()}")
rec.head(3)

In [ ]:
# Target: recurring monthly total spend per proponent
ONE_OFF_EUR = 1000.0

one_off = rec["Amount"] >= ONE_OFF_EUR
print(f"one-offs dropped: {one_off.sum()} receipts, "
      f"{rec.loc[one_off, 'Amount'].sum():,.0f} EUR "
      f"({rec.loc[one_off, 'Amount'].sum() / rec['Amount'].sum() * 100:.1f}% of spend)")

span = rec.groupby("uuid")["month"].agg(lambda m: (m.max() - m.min()).n + 1).rename("span_months")
kept_total = rec[~one_off].groupby("uuid")["Amount"].sum().rename("kept_total")

target = pd.concat([span, kept_total], axis=1).fillna({"kept_total": 0.0}).reset_index()
target["monthly_expenses"] = (target["kept_total"] / target["span_months"]).round(2)

print(f"\ntarget: {len(target)} proponents")
print("\nmonthly_expenses (EUR):")
print(target["monthly_expenses"].describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())

In [ ]:
# Month-to-month variation: how far actual monthly spend swings around the average.
# Build each proponent's monthly series over their full span (empty months = 0), then:
#   monthly_std = std of the monthly totals (EUR)
#   monthly_cv  = monthly_std / monthly_expenses (scale-free: 0.45 = swings ~45% of the average)
kept_by_month = rec[~one_off].groupby(["uuid", "month"])["Amount"].sum()
bounds = rec.groupby("uuid")["month"].agg(["min", "max"])

monthly_std = {
    u: kept_by_month.loc[u].reindex(pd.period_range(m0, m1, freq="M"), fill_value=0.0).std()
    for u, (m0, m1) in bounds.iterrows()
}
target["monthly_std"] = target["uuid"].map(monthly_std).round(2)
target["monthly_cv"] = (target["monthly_std"] / target["monthly_expenses"]).round(3)

print("monthly_std (EUR):")
print(target["monthly_std"].describe(percentiles=[.25, .5, .75, .9]).round(1).to_string())
print("\nmonthly_cv:")
print(target["monthly_cv"].describe(percentiles=[.25, .5, .75, .9]).round(2).to_string())
target.head(5)

In [ ]:
# Sector shares: where the recurring spend goes. Same 900 EUR/month means something
# different at 40% health vs 40% dining. Native sectors map to broad buckets;
# "Outros" -> other_retail, null -> uncategorized (together ~88% of spend, so the
# named shares are small but they are the discriminating ones).
SECTOR_MAP = {
    "Alojamento, restauração e similares": "food_dining",
    "Saúde": "health", "Atividades veterinárias": "health", "Ginásios": "health", "Lares": "health",
    "Manutenção e reparação de veículos automóveis": "transport",
    "Manutenção e reparação de motociclos, de suas peças e acessórios": "transport",
    "Aquisição de passes mensais ou de bilhetes para utilização de transportes públicos coletivos": "transport",
    "Educação": "education", "Comércio a retalho de livros": "education", "Jornais e Revistas": "education",
    "Atividades de salões de cabeleireiro e institutos de beleza": "personal_care",
    "Imóveis": "housing",
    "Atividades artísticas e literárias": "leisure",
}
CATEGORIES = ["food_dining", "health", "transport", "education", "housing",
              "personal_care", "leisure", "other_retail", "uncategorized"]

rec["category"] = rec["Sector"].map(
    lambda s: "uncategorized" if pd.isna(s) else SECTOR_MAP.get(s, "other_retail")
)

shares = (
    rec[~one_off]
    .pivot_table(index="uuid", columns="category", values="Amount", aggfunc="sum", fill_value=0.0)
    .reindex(columns=CATEGORIES, fill_value=0.0)
)
shares = shares.div(shares.sum(axis=1), axis=0).round(4).add_prefix("share_")

for c in shares.columns:
    target[c] = target["uuid"].map(shares[c])

print("share stats across proponents (%):")
print((target[shares.columns].describe(percentiles=[.5, .9]).T[["mean", "50%", "90%", "max"]] * 100)
      .round(1).sort_values("mean", ascending=False).to_string())
target.head(3)

In [ ]:
# Expenses over 1000 EUR: the receipts excluded from the recurring target are a
# signal in themselves (durables propensity / lumpy outgoings).
#   expenses_over_1000_count = number of such receipts in the observed window
#   expenses_over_1000_total = their summed value (EUR, window total, NOT monthly)
#   expenses_over_1000_share = total / raw total spend (0 = never, 0.7 = most spend is lumpy)
raw_total = rec.groupby("uuid")["Amount"].sum()
over = rec[one_off].groupby("uuid")["Amount"].agg(count="count", total="sum")

target["expenses_over_1000_count"] = target["uuid"].map(over["count"]).fillna(0).astype(int)
target["expenses_over_1000_total"] = target["uuid"].map(over["total"]).fillna(0.0).round(2)
target["expenses_over_1000_share"] = (
    target["expenses_over_1000_total"] / target["uuid"].map(raw_total)
).round(4)

cols = ["expenses_over_1000_count", "expenses_over_1000_total", "expenses_over_1000_share"]
print(f"proponents with at least one receipt over 1000 EUR: "
      f"{(target.expenses_over_1000_count > 0).sum()} of {len(target)}")
print(target[cols].describe(percentiles=[.5, .75, .9]).round(2).to_string())
target.head(3)

In [ ]:
import matplotlib.pyplot as plt

INK, INK2, BLUE = "#0b0b0b", "#52514e", "#2a78d6"

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(target["monthly_expenses"], bins=18, color=BLUE, edgecolor="white", linewidth=1)

med = target["monthly_expenses"].median()
ax.axvline(med, color=INK2, ls="--", lw=1)
ax.text(med * 1.03, ax.get_ylim()[1] * 0.95, f"median {med:,.0f} EUR",
        color=INK2, fontsize=9, va="top")

ax.set_title("Recurring monthly expenses per proponent (e-fatura, one-offs removed)",
             color=INK, fontsize=11, loc="left")
ax.set_xlabel("EUR / month", color=INK2)
ax.set_ylabel("proponents", color=INK2)
ax.tick_params(colors=INK2, labelsize=9)
ax.grid(axis="y", color="#e6e5e0", lw=0.6)
ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
plt.tight_layout()
plt.show()

In [ ]:
target.head()

### 4.2 Feature table

One row per proponent, joined with `target` into `dataset`. Sources:

- **Income**: `irs_income` = annual `IRSTaxDeclarationDetails.Total.Income`, falling back to the previous-year declaration (43/45). `avg_regular_income_6m` from `CalculatedIncomes` is fresher but exists for only 20/45, so it ships with a presence flag.
- **Household**: `n_dependents` (empty list = 0), `is_couple` (MaritalStatus in {Casado, Unido de facto}).
- **Demographics**: `age` (`PersonalTaxData.BirthDate` already holds an anonymized age, not a date).
- **Debt (current CRC)**: `crc_total_installment`, `crc_n_active` (liabilities with TotalDebt > 0), `crc_has_mortgage` ("Crédito à habitação").
- **Housing**: `is_renter` (non-empty `RenterContractList`).

The spend-profile columns built in 4.1 (`monthly_std`, `monthly_cv`, `share_*`, `expenses_over_1000_*`) are carried into `dataset` as features too. **Caveat**: they derive from the same e-fatura receipts as the target, so any model using them only applies to proponents with e-fatura coverage, and they are partly circular for predicting `monthly_expenses` itself.

In [ ]:
# Proponent-side features, built from the parsed profiles (dicts), one row per proponent.
COUPLE = {"Casado", "Unido de facto"}
MORTGAGE = "Crédito à habitação"

def dig(d, *path):
    for k in path:
        d = d.get(k) if isinstance(d, dict) else None
    return d

feat_rows = []
for uuid, p in zip(prop_raw["uuid"], profiles):
    cur = p.get("IRSTaxDeclarationDetails") or {}
    prev = p.get("IRSTaxDeclarationDetailsPreviousYear") or {}
    irs = dig(cur, "Total", "Income")
    if irs is None:
        irs = dig(prev, "Total", "Income")
    marital = cur.get("MaritalStatus") or prev.get("MaritalStatus")
    liabs = dig(p, "CRCData", "Liabilities") or []
    feat_rows.append({
        "uuid": uuid,
        "irs_income": irs,
        "avg_regular_income_6m": dig(p, "CalculatedIncomes", "AVGRegularIncome6Month"),
        "n_dependents": len(cur.get("Dependents") or []),
        "is_couple": None if marital is None else int(marital in COUPLE),
        "age": dig(p, "PersonalTaxData", "BirthDate"),
        "crc_total_installment": round(sum((l.get("Installment") or 0) for l in liabs), 2),
        "crc_n_active": sum(1 for l in liabs if (l.get("TotalDebt") or 0) > 0),
        "crc_has_mortgage": int(any(l.get("Product") == MORTGAGE for l in liabs)),
        "is_renter": int(bool(p.get("RenterContractList"))),
    })

features = pd.DataFrame(feat_rows)
features["has_avg_income_6m"] = features["avg_regular_income_6m"].notna().astype(int)

print(f"features: {features.shape[0]} proponents x {features.shape[1] - 1} features")
print("\nmissing per column:")
print(features.isna().sum()[lambda s: s > 0].to_string())
features.drop(columns="uuid").describe().round(1)

In [ ]:
# Modeling dataset: features + spend profile + target, one row per proponent.
dataset = features.merge(target, on="uuid", how="inner")

TARGET_COL = "monthly_expenses"
NOT_FEATURES = ["uuid", "kept_total", "span_months", TARGET_COL]
FEATURE_COLS = [c for c in dataset.columns if c not in NOT_FEATURES]

assert len(dataset) == len(target)
print(f"dataset: {dataset.shape[0]} proponents x {dataset.shape[1]} columns")
print(f"target: {TARGET_COL}")
print(f"{len(FEATURE_COLS)} feature columns:")
print("  proponent-side:", [c for c in FEATURE_COLS if c in features.columns])
print("  spend-profile: ", [c for c in FEATURE_COLS if c not in features.columns])
dataset.head(3)

### 4.3 Model

**Step 1: signal check.** Spearman correlation of every feature against `monthly_expenses` (rank-based, so robust to the skew and the 4,400 EUR outlier), before fitting anything.

In [ ]:
corr = (
    dataset[FEATURE_COLS + [TARGET_COL]]
    .corr(method="spearman")[TARGET_COL]
    .drop(TARGET_COL)
)
signal = pd.DataFrame({
    "spearman": corr.round(2),
    "n": dataset[FEATURE_COLS].notna().sum(),
    "side": ["proponent" if c in features.columns else "spend" for c in corr.index],
})
signal.sort_values("spearman", key=abs, ascending=False)

In [ ]:
# Full Spearman correlation matrix: target + every feature, sorted so the
# variables most related to monthly_expenses come first (top-left).
# Blue = positive, red = negative, gray = no relationship.
from matplotlib.colors import LinearSegmentedColormap

cm_cols = [TARGET_COL] + [c for c in FEATURE_COLS if dataset[c].std() > 0]  # drops share_leisure (all zeros)
cm = dataset[cm_cols].corr(method="spearman")
order = [TARGET_COL] + cm[TARGET_COL].drop(TARGET_COL).abs().sort_values(ascending=False).index.tolist()
cm = cm.loc[order, order]

DIVERGING = LinearSegmentedColormap.from_list("corr", ["#e34948", "#f0efec", "#2a78d6"])

fig, ax = plt.subplots(figsize=(10.5, 9))
im = ax.imshow(cm.values, cmap=DIVERGING, vmin=-1, vmax=1)
ax.set_xticks(range(len(cm)))
ax.set_xticklabels(cm.columns, rotation=90, fontsize=8, color=INK2)
ax.set_yticks(range(len(cm)))
ax.set_yticklabels(cm.index, fontsize=8, color=INK2)
cbar = fig.colorbar(im, ax=ax, shrink=0.7)
cbar.set_label("spearman rho", color=INK2)
cbar.ax.tick_params(colors=INK2, labelsize=8)
ax.set_title("Correlation matrix (sorted by strength against the target)",
             loc="left", color=INK, fontsize=11)
plt.tight_layout()
plt.show()

cm.round(2)

In [ ]:
# The two strongest proponent-side features, against the target (log-log: both are skewed).
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, xcol, xlabel in [
    (axes[0], "irs_income", "IRS annual income (EUR, log)"),
    (axes[1], "crc_total_installment", "CRC total monthly installment (EUR, log)"),
]:
    d = dataset.dropna(subset=[xcol])
    d = d[d[xcol] > 0]
    ax.scatter(d[xcol], d[TARGET_COL], s=42, color=BLUE, alpha=0.75, edgecolors="white", linewidths=0.8)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(xlabel, color=INK2)
    ax.grid(color="#e6e5e0", lw=0.6); ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, labelsize=9)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#c3c2b7")
    rho = d[xcol].corr(d[TARGET_COL], method="spearman")
    ax.set_title(f"spearman {rho:.2f}", color=INK2, fontsize=9, loc="right")
axes[0].set_ylabel("monthly_expenses (EUR, log)", color=INK2)
fig.suptitle("Income and debt service vs recurring monthly expenses", color=INK, fontsize=11, x=0.01, ha="left")
plt.tight_layout()
plt.show()

**Step 2: fit and evaluate.**

**Train/test design**: the 47 proponents dropped by the `created-2` filter are usable as a test set. Only their *historical CRC pulls* came from the bad extraction; their current snapshot (`value`: e-fatura, IRS, current CRC) is the same quality as everyone else's, and neither the target nor any model feature touches the historical pulls. So: **train = 45 clean-batch proponents, test = 47 early-batch proponents**. Not a random split (it's by extraction batch), but there is no reason to expect the cohorts to differ, and it gives an out-of-sample set larger than the train set.

**Model**: Ridge regression on `log(monthly_expenses)` with the 5 features that showed signal and exist for (almost) everyone: `irs_income`, `crc_total_installment`, `crc_has_mortgage`, `is_couple`, `age`. Median imputation for the few missing values, standardized inputs, regularization strength chosen by leave-one-out on train. Baseline to beat: always predict the train median.

In [ ]:
import numpy as np

MODEL_FEATURES = ["irs_income", "crc_total_installment", "crc_has_mortgage", "is_couple", "age"]

def build_model_frame(snap):
    """Target + the 5 model features from a (uuid, value) snapshot frame.
    Reproduces the 4.1/4.2 construction for any batch of proponents."""
    rows = []
    for uuid, p in zip(snap["uuid"], snap["value"].map(json.loads)):
        exp = p.get("ExpensesList") or []
        if exp:
            amounts = np.array([e["Amount"] for e in exp], dtype=float)
            months = pd.PeriodIndex(
                pd.to_datetime([e["IssuanceDate"] for e in exp], utc=True, format="mixed")
                .tz_localize(None), freq="M",
            )
            span = (months.max() - months.min()).n + 1
            monthly = amounts[amounts < ONE_OFF_EUR].sum() / span
        else:
            monthly = np.nan  # no receipts: no target; row gets dropped for modeling
        cur = p.get("IRSTaxDeclarationDetails") or {}
        prev = p.get("IRSTaxDeclarationDetailsPreviousYear") or {}
        irs = dig(cur, "Total", "Income")
        if irs is None:
            irs = dig(prev, "Total", "Income")
        marital = cur.get("MaritalStatus") or prev.get("MaritalStatus")
        liabs = dig(p, "CRCData", "Liabilities") or []
        rows.append({
            "uuid": uuid,
            "monthly_expenses": monthly,
            "irs_income": irs,
            "crc_total_installment": sum((l.get("Installment") or 0) for l in liabs),
            "crc_has_mortgage": int(any(l.get("Product") == MORTGAGE for l in liabs)),
            "is_couple": None if marital is None else int(marital in COUPLE),
            "age": dig(p, "PersonalTaxData", "BirthDate"),
        })
    return pd.DataFrame(rows)

# Test set: the 47 proponents dropped by the created-2 filter (see markdown above).
raw_all = pd.read_csv("../data/data-1786540369000.csv")
raw_all["created-2"] = pd.to_datetime(raw_all["created-2"])
test_snap = (
    raw_all[~raw_all["uuid"].isin(set(dataset["uuid"]))]
    .drop_duplicates("uuid")[["uuid", "value"]]
    .reset_index(drop=True)
)
test_df = build_model_frame(test_snap)

# Sanity: the same function applied to the train snapshots must reproduce `dataset`.
train_df = build_model_frame(prop_raw[["uuid", "value"]])
merged = train_df.merge(dataset, on="uuid", suffixes=("_fn", ""))
assert np.allclose(merged["monthly_expenses_fn"], merged["monthly_expenses"], atol=0.01)

print(f"train: {len(train_df)} proponents | test: {len(test_df)} proponents (no overlap)")
print(f"test missing: irs_income {test_df['irs_income'].isna().sum()}, "
      f"is_couple {test_df['is_couple'].isna().sum()}")
test_df.head(3)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_train = train_df.merge(dataset[["uuid"]], on="uuid")[MODEL_FEATURES].astype(float)
y_train = np.log(dataset.set_index("uuid").loc[train_df["uuid"], TARGET_COL]).values
X_test = test_df[MODEL_FEATURES].astype(float)
y_test = np.log(test_df["monthly_expenses"]).values

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RidgeCV(alphas=np.logspace(-2, 3, 60)),  # alpha chosen by efficient LOO on train
)

# Honest train-side error: leave-one-out predictions, each point predicted by a model
# that never saw it.
loo_pred = cross_val_predict(model, X_train, y_train, cv=LeaveOneOut())
model.fit(X_train, y_train)
test_pred = model.predict(X_test)

def report(name, y_log, p_log, baseline_log):
    y, p = np.exp(y_log), np.exp(p_log)
    baseline = np.exp(np.full_like(y_log, baseline_log))
    mae, base_mae = np.mean(np.abs(y - p)), np.mean(np.abs(y - baseline))
    med_ape = np.median(np.abs(y - p) / y) * 100
    r2 = 1 - np.sum((y_log - p_log) ** 2) / np.sum((y_log - y_log.mean()) ** 2)
    print(f"{name}: MAE {mae:.0f} EUR (median-baseline {base_mae:.0f}), "
          f"median abs error {med_ape:.0f}%, R2(log) {r2:.2f}")

baseline_log = np.median(y_train)
report("train (LOO)", y_train, loo_pred, baseline_log)
report("test  (47) ", y_test, test_pred, baseline_log)

print(f"\nchosen alpha: {model[-1].alpha_:.1f}")
print("\nstandardized coefficients (effect on log expenses, per 1 sd of the feature):")
coefs = pd.Series(model[-1].coef_, index=MODEL_FEATURES)
print(coefs.sort_values(key=abs, ascending=False).round(3).to_string())

## 5. Full dataset: the same model on 565 proponents

`data_full.csv` is the complete extraction (it contains the 92 proponents from the partial file). Its *historical CRC pulls* carry the same extraction bug for everything created before 2026-08-07 (936 of 1,053 pulls are copies of the current CRC), but **this model never touches them**: target and features come only from the `value` snapshot, which is clean in every row.

Design changes vs section 4:

- Population: 565 proponents; 3 have no e-fatura receipts and are dropped (no target), leaving **562**.
- Split: the extraction-batch split is retired; standard **random 70/30 train/test** (seed 42), with 5-fold CV on the train side for the honest train error.
- Same 5 features, same Ridge-on-log pipeline, for comparability with section 4.

In [ ]:
# Build the full-population modeling frame (snapshots only; takes ~15s to parse 565 JSONs).
full_raw = pd.read_csv("../data/data_full.csv")
snap_full = full_raw.drop_duplicates("uuid")[["uuid", "value"]].reset_index(drop=True)
full_df = build_model_frame(snap_full)

usable = full_df["monthly_expenses"] > 0  # drops no-receipt rows (NaN) and all-one-off rows (0)
print(f"{len(snap_full)} proponents parsed, {(~usable).sum()} without a usable target -> "
      f"{usable.sum()} modeling rows")
full_df = full_df[usable].reset_index(drop=True)

print("\nmonthly_expenses (EUR):")
print(full_df["monthly_expenses"].describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())

print("\nspearman vs target (n=562):")
sp = (full_df[MODEL_FEATURES + ["monthly_expenses"]]
      .corr(method="spearman")["monthly_expenses"].drop("monthly_expenses"))
print(sp.sort_values(key=abs, ascending=False).round(2).to_string())

In [ ]:
from sklearn.model_selection import KFold, train_test_split

X_full = full_df[MODEL_FEATURES].astype(float)
y_full = np.log(full_df["monthly_expenses"])

X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_full, test_size=0.3, random_state=42)

model_full = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RidgeCV(alphas=np.logspace(-2, 3, 60)),
)

cv_pred = cross_val_predict(model_full, X_tr, y_tr, cv=KFold(5, shuffle=True, random_state=42))
model_full.fit(X_tr, y_tr)
te_pred = model_full.predict(X_te)

baseline_log = np.median(y_tr)
report("train (5-fold CV, n=393)", y_tr.values, cv_pred, baseline_log)
report("test  (30% held, n=169) ", y_te.values, te_pred, baseline_log)

print(f"\nchosen alpha: {model_full[-1].alpha_:.1f}")
print("\nstandardized coefficients (effect on log expenses, per 1 sd of the feature):")
print(pd.Series(model_full[-1].coef_, index=MODEL_FEATURES)
      .sort_values(key=abs, ascending=False).round(3).to_string())

In [ ]:
# Predicted vs actual on the held-out test proponents.
actual = np.exp(y_te.values)
pred = np.exp(te_pred)

fig, ax = plt.subplots(figsize=(7.2, 6.4))
lo = min(actual.min(), pred.min()) * 0.85
hi = max(actual.max(), pred.max()) * 1.15
lims = (lo, hi)
ax.plot(lims, lims, color=INK2, lw=1)
ax.plot(lims, [l * 1.5 for l in lims], color="#c3c2b7", lw=0.9, ls="--")
ax.plot(lims, [l / 1.5 for l in lims], color="#c3c2b7", lw=0.9, ls="--")
ax.scatter(actual, pred, s=40, color=BLUE, alpha=0.65, edgecolors="white", linewidths=0.7)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("actual monthly_expenses (EUR, log)", color=INK2)
ax.set_ylabel("predicted (EUR, log)", color=INK2)
within = np.mean(np.abs(pred - actual) / actual <= 0.5) * 100
ax.set_title(f"Predicted vs actual, {len(actual)} test proponents "
             f"({within:.0f}% within +/-50%, solid line = perfect)",
             loc="left", color=INK, fontsize=11)
ax.grid(color="#e6e5e0", lw=0.6); ax.set_axisbelow(True)
ax.tick_params(colors=INK2, labelsize=9)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color("#c3c2b7")
plt.tight_layout()
plt.show()

# Where it misses: median predicted/actual ratio by actual-spend quartile.
ratio_by_q = (pd.DataFrame({"actual": actual, "ratio": pred / actual})
              .groupby(pd.qcut(actual, 4, labels=["q1 low", "q2", "q3", "q4 high"]), observed=True)["ratio"]
              .agg(["median", "count"]).round(2))
print(ratio_by_q.to_string())

## 6. Default labels from the current CRC

Goal: a supervised look at whether spending behaviour separates proponents who default. Labels come from the current CRC liabilities (`TotalDefault`, `DefaultInitialDate`):

- `default_any`: TotalDefault > 0 on any liability.
- `default_significant`: summed TotalDefault > 500 EUR (filters technical arrears like a missed card payment).
- `first_default`: earliest `DefaultInitialDate` among liabilities currently in default.

**Population rules** (each proponent's e-fatura window is ~12 months ending at their snapshot creation):

1. No receipts: excluded (no behaviour to observe).
2. `first_default` **before** the window: excluded; their observed spending is entirely post-default.
3. `first_default` **inside** the window: kept, but the observation window is **truncated at the month before `first_default`**, so behaviour features will describe only pre-default months.
4. After truncation, defaulters need at least `MIN_OBS_MONTHS` (3) observed months, otherwise excluded (a 1-2 month behaviour profile is noise).

**Honest scope note**: this is still current-status association (defaulters observed shortly before their default), not a point-in-time PD model. The question it can answer: does pre-default spending behaviour look different from non-defaulter behaviour?

In [ ]:
MIN_OBS_MONTHS = 3
DEFAULT_SIGNIFICANT_EUR = 500.0

label_rows = []
for uuid, p in zip(snap_full["uuid"], snap_full["value"].map(json.loads)):
    exp = p.get("ExpensesList") or []
    months = (pd.to_datetime([e["IssuanceDate"] for e in exp], utc=True, format="mixed")
              .tz_localize(None).to_period("M")) if exp else None
    liabs = (p.get("CRCData") or {}).get("Liabilities") or []
    in_def = [l for l in liabs if (l.get("TotalDefault") or 0) > 0]
    dates = [pd.Timestamp(l["DefaultInitialDate"]).to_period("M")
             for l in in_def if l.get("DefaultInitialDate")]
    label_rows.append({
        "uuid": uuid,
        "win_start": months.min() if months is not None else None,
        "win_end": months.max() if months is not None else None,
        "total_default": round(sum((l.get("TotalDefault") or 0) for l in in_def), 2),
        "first_default": min(dates) if dates else None,
    })
labels = pd.DataFrame(label_rows)
labels["default_any"] = labels["total_default"] > 0
labels["default_significant"] = labels["total_default"] > DEFAULT_SIGNIFICANT_EUR

# Observation window: defaulters are truncated at the month before first_default.
labels["obs_end"] = [
    min(e, f - 1) if pd.notna(f) and pd.notna(e) else e
    for e, f in zip(labels["win_end"], labels["first_default"])
]
labels["obs_months"] = [
    max((e - s).n + 1, 0) if pd.notna(s) and pd.notna(e) else 0
    for s, e in zip(labels["win_start"], labels["obs_end"])
]

no_receipts = labels["win_start"].isna()
pre_window = pd.Series(
    [pd.notna(f) and pd.notna(s) and f < s
     for f, s in zip(labels["first_default"], labels["win_start"])],
    index=labels.index,
)
too_short = labels["default_any"] & ~pre_window & (labels["obs_months"] < MIN_OBS_MONTHS)
labels["usable"] = ~(no_receipts | pre_window | too_short)

print(f"population {len(labels)} | excluded: {no_receipts.sum()} no receipts, "
      f"{pre_window.sum()} defaulted pre-window, {too_short.sum()} under {MIN_OBS_MONTHS} pre-default months")
kept = labels[labels["usable"]]
print(f"usable: {len(kept)} proponents")
print(f"  default_any:         {kept['default_any'].sum()} ({kept['default_any'].mean()*100:.1f}%)")
print(f"  default_significant: {kept['default_significant'].sum()} ({kept['default_significant'].mean()*100:.1f}%)")
print(f"  obs months, defaulters kept: median "
      f"{kept.loc[kept['default_any'], 'obs_months'].median():.0f} "
      f"(non-defaulters: {kept.loc[~kept['default_any'], 'obs_months'].median():.0f})")
labels[labels["default_any"]].head(5)

**Step 2: behaviour features on the truncated window, and whether they separate defaulters.**

Each proponent gets a behaviour profile computed **only on their observation window**:

- window = e-fatura months, **minus the first and last month** (both are partial months: the window starts at the first receipt and ends at snapshot creation mid-month; keeping them deflates edge months and inflates volatility, and unevenly so, since defaulters' truncated windows end on a full month), then truncated at the month before `first_default` for defaulters. Minimum 3 full months.
- `actual_expenses` (recurring monthly, as in 4.1), `monthly_cv`, `one_off_share`: behaviour.
- `expected_expenses`: the section-5 model form applied **out-of-fold** (5-fold; each proponent is scored by a model that never saw them), and `overspend_ratio` = actual / expected.
- capacity metrics: `monthly_income` (IRS/12), `dsti` (installments/income), `eti` (expenses/income), `residual_income` (income - installments - expenses).

Then the discrimination check: median per group and univariate AUC of each metric against both labels. AUC 0.5 = no signal; "inv" = defaulters have *lower* values.

In [ ]:
# Behaviour profile per usable proponent, on the trimmed + truncated window.
lab = labels.set_index("uuid")
usable_ids = set(labels.loc[labels["usable"], "uuid"])

behav_rows, too_short_trim = [], 0
for uuid, p in zip(snap_full["uuid"], snap_full["value"].map(json.loads)):
    if uuid not in usable_ids:
        continue
    L = lab.loc[uuid]
    start = L["win_start"] + 1                      # drop partial first month
    end = min(L["win_end"] - 1, L["obs_end"])       # drop partial last month, keep truncation
    n_months = (end - start).n + 1
    if n_months < MIN_OBS_MONTHS:
        too_short_trim += 1
        continue
    exp = p.get("ExpensesList") or []
    r = pd.DataFrame({
        "amount": [e["Amount"] for e in exp],
        "month": pd.to_datetime([e["IssuanceDate"] for e in exp], utc=True, format="mixed")
                 .tz_localize(None).to_period("M"),
    })
    r = r[(r["month"] >= start) & (r["month"] <= end)]
    kept = r[r["amount"] < ONE_OFF_EUR]
    series = (kept.groupby("month")["amount"].sum()
              .reindex(pd.period_range(start, end, freq="M"), fill_value=0.0))
    actual = kept["amount"].sum() / n_months
    over_total = r.loc[r["amount"] >= ONE_OFF_EUR, "amount"].sum()
    cur = p.get("IRSTaxDeclarationDetails") or {}
    prev = p.get("IRSTaxDeclarationDetailsPreviousYear") or {}
    irs = dig(cur, "Total", "Income")
    if irs is None:
        irs = dig(prev, "Total", "Income")
    marital = cur.get("MaritalStatus") or prev.get("MaritalStatus")
    liabs = dig(p, "CRCData", "Liabilities") or []
    behav_rows.append({
        "uuid": uuid,
        "obs_months_trimmed": n_months,
        "actual_expenses": round(actual, 2),
        "monthly_cv": round(series.std() / actual, 3) if actual > 0 else np.nan,
        "one_off_share": round(over_total / r["amount"].sum(), 4) if r["amount"].sum() > 0 else 0.0,
        "irs_income": irs,
        "crc_total_installment": round(sum((l.get("Installment") or 0) for l in liabs), 2),
        "crc_has_mortgage": int(any(l.get("Product") == MORTGAGE for l in liabs)),
        "is_couple": None if marital is None else int(marital in COUPLE),
        "age": dig(p, "PersonalTaxData", "BirthDate"),
    })

behav = pd.DataFrame(behav_rows)
zero_spend = behav["actual_expenses"] <= 0
behav = behav[~zero_spend].reset_index(drop=True)
print(f"behaviour profiles: {len(behav)} proponents "
      f"({too_short_trim} lost to trimming, {zero_spend.sum()} with zero kept spend)")
behav.head(3)

In [ ]:
from sklearn.metrics import roc_auc_score

# Expected expenses for each profile, out-of-fold (nobody scored by a model that saw them).
oof_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RidgeCV(alphas=np.logspace(-2, 3, 60)),
)
oof = cross_val_predict(
    oof_pipe,
    behav[MODEL_FEATURES].astype(float),
    np.log(behav["actual_expenses"]),
    cv=KFold(5, shuffle=True, random_state=42),
)
behav["expected_expenses"] = np.exp(oof).round(2)
behav["overspend_ratio"] = (behav["actual_expenses"] / behav["expected_expenses"]).round(3)
behav["monthly_income"] = (behav["irs_income"] / 12).round(2)
behav.loc[behav["monthly_income"] <= 0, "monthly_income"] = np.nan  # zero income: ratios undefined
behav["dsti"] = (behav["crc_total_installment"] / behav["monthly_income"]).round(3)
behav["eti"] = (behav["actual_expenses"] / behav["monthly_income"]).round(3)
behav["residual_income"] = (behav["monthly_income"] - behav["crc_total_installment"]
                            - behav["actual_expenses"]).round(2)

behav_l = behav.merge(labels[["uuid", "default_any", "default_significant"]], on="uuid")

BEHAV_METRICS = ["actual_expenses", "expected_expenses", "overspend_ratio", "monthly_cv",
                 "one_off_share", "monthly_income", "dsti", "eti", "residual_income"]

print("medians, non-defaulters vs defaulters (default_any):")
print(behav_l.groupby("default_any")[BEHAV_METRICS].median().T.round(2).to_string())

print("\nunivariate AUC (0.5 = no signal; inv = defaulters LOWER):")
print(f"{'metric':22s} {'vs any':>10s} {'vs significant':>16s}   n")
for m in BEHAV_METRICS:
    d = behav_l.dropna(subset=[m])
    row = []
    for lbl in ["default_any", "default_significant"]:
        auc = roc_auc_score(d[lbl], d[m])
        row.append(f"{max(auc, 1 - auc):.3f}{' inv' if auc < 0.5 else '    '}")
    print(f"{m:22s} {row[0]:>10s} {row[1]:>16s}   {len(d)}")

**Step 3: combined behaviour score (honest negative result).**

Logistic regression on the three surviving metrics (`monthly_cv`, `dsti`, `residual_income`), scored out-of-fold. With only ~24 significant defaulters among rows with income, the fitted combination does **not** beat the best single metric out-of-fold, and the quintile lift is not monotone. Conclusion for now: the individual signals are real but weak-to-moderate, and 24-28 positives cannot support learned weights. Until more outcomes are available, an expert scorecard on the univariate signals (flags like residual_income < 0, dsti > 0.4, low monthly_cv) is the defensible way to use them.

In [ ]:
# Combined behaviour score: logistic on the three surviving metrics, evaluated
# strictly out-of-fold (stratified 5-fold; every proponent scored by a model
# that never saw them).
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

SCORE_FEATURES = ["monthly_cv", "dsti", "residual_income"]
d_score = behav_l.dropna(subset=SCORE_FEATURES).reset_index(drop=True)
X_score = d_score[SCORE_FEATURES].astype(float)
print(f"rows: {len(d_score)} | default_any: {d_score['default_any'].sum()} | "
      f"default_significant: {d_score['default_significant'].sum()}")

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
oof_sig = None
for lbl in ["default_significant", "default_any"]:
    y = d_score[lbl].astype(int)
    oof = cross_val_predict(clf, X_score, y,
                            cv=StratifiedKFold(5, shuffle=True, random_state=42),
                            method="predict_proba")[:, 1]
    print(f"{lbl}: OOF AUC {roc_auc_score(y, oof):.3f}")
    if lbl == "default_significant":
        oof_sig = oof

clf.fit(X_score, d_score["default_significant"].astype(int))
print("\ncoefficients (standardized):")
print(pd.Series(clf[-1].coef_[0], index=SCORE_FEATURES).round(3).to_string())

# Lift check: default_significant rate by OOF-score quintile (should rise monotonically
# for a usable score).
quintile = pd.qcut(oof_sig, 5, labels=["q1 safest", "q2", "q3", "q4", "q5 riskiest"])
print("\ndefault_significant rate by score quintile:")
print(pd.DataFrame({"y": d_score["default_significant"].astype(int), "q": quintile})
      .groupby("q", observed=True)["y"].agg(["mean", "sum", "count"]).round(3).to_string())